In [1]:
import urllib.request
import datetime
import os

# Створюємо папку для збереження даних, якщо її ще немає
if not os.path.exists('vhi_data'):
    os.makedirs('vhi_data')

def download_noaa_data():
    print("Починаємо завантаження...")
    # Від 1 до 27, бо область 0
    for province_id in range(1, 28): 
        # Перевіряємо, чи вже є файл для цієї області 
        existing_files = [f for f in os.listdir('vhi_data') if f.startswith(f"vhi_id_{province_id}_")]
        if existing_files:
            print(f"Дані для області {province_id} вже існують. Пропускаємо.")
            continue

        # Формуємо посилання (1981-2024)
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"

        try:
            # Завантажуємо
            vhi_url = urllib.request.urlopen(url)
            text = vhi_url.read().decode('utf-8')

            # Прибираємо HTML теги які сайт віддає разом з текстом
            text = text.replace("<tt><pre>", "").replace("</pre></tt>", "")

            # Додаємо поточну дату та час до назви файлу
            now = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            filename = f"vhi_data/vhi_id_{province_id}_{now}.csv"

            # Зберігаємо
            with open(filename, 'w') as file:
                file.write(text)
            print(f"Збережено: {filename}")
        except Exception as e:
            print(f"Помилка для області {province_id}: {e}")
            
    print("Завантаження завершено!")

download_noaa_data()

Починаємо завантаження...
Збережено: vhi_data/vhi_id_1_2026-03-18_13-43-31.csv
Збережено: vhi_data/vhi_id_2_2026-03-18_13-43-32.csv
Збережено: vhi_data/vhi_id_3_2026-03-18_13-43-34.csv
Збережено: vhi_data/vhi_id_4_2026-03-18_13-43-35.csv
Збережено: vhi_data/vhi_id_5_2026-03-18_13-43-36.csv
Збережено: vhi_data/vhi_id_6_2026-03-18_13-43-37.csv
Збережено: vhi_data/vhi_id_7_2026-03-18_13-43-38.csv
Збережено: vhi_data/vhi_id_8_2026-03-18_13-43-39.csv
Збережено: vhi_data/vhi_id_9_2026-03-18_13-43-40.csv
Збережено: vhi_data/vhi_id_10_2026-03-18_13-43-41.csv
Збережено: vhi_data/vhi_id_11_2026-03-18_13-43-42.csv
Збережено: vhi_data/vhi_id_12_2026-03-18_13-43-43.csv
Збережено: vhi_data/vhi_id_13_2026-03-18_13-43-44.csv
Збережено: vhi_data/vhi_id_14_2026-03-18_13-43-45.csv
Збережено: vhi_data/vhi_id_15_2026-03-18_13-43-46.csv
Збережено: vhi_data/vhi_id_16_2026-03-18_13-43-47.csv
Збережено: vhi_data/vhi_id_17_2026-03-18_13-43-48.csv
Збережено: vhi_data/vhi_id_18_2026-03-18_13-43-49.csv
Збережено: 

In [2]:
import pandas as pd
import glob
import os

def create_dataframe_and_clean():
    print("Починаємо збирання та очищення даних...")
    
    # Знаходимо всі файли .csv у нашій папці
    all_files = glob.glob('vhi_data/*.csv')
    df_list = []
    
    # Словник для заміни індексів NOAA на український стандарт
    province_mapping = {
        1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21, 
        11: 9, 12: 0, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16, 
        20: 0, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
    }
    
    for file in all_files:
        # Витягуємо старий ID області з назви файлу
        file_name = os.path.basename(file)
        old_province_id = int(file_name.split('_')[2])
        
        # Читаємо файл, ігноруємо перший рядок-заголовок, задаємо свої імена колонкам
        df_temp = pd.read_csv(file, header=1, names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty'])
        df_temp['Area_ID'] = old_province_id
        df_list.append(df_temp)
        
    # Зліплюємо всі 27 таблиць в одну велику
    df = pd.concat(df_list, ignore_index=True)
    
    # Data Cleaning
    df = df.drop(columns=['empty']) 
    df = df.dropna() 
    
    # Очищаємо колонку Year від HTML-тегів, які могли залишитися
    df['Year'] = df['Year'].astype(str).str.replace('<br>', '').str.strip()
    df = df[df['Year'] != ''] # Прибираємо пусті значення, якщо вони виникли
    
    # Конвертуємо типи даних
    df['Year'] = df['Year'].astype(int)
    df['Week'] = df['Week'].astype(int)
    df['VHI'] = df['VHI'].astype(float)
    
    # Видаляємо рядки, де VHI = -1
    df = df[df['VHI'] != -1.0]
    
    # Змінюємо старі індекси на нові за допомогою нашого словника
    df['Area_ID'] = df['Area_ID'].map(province_mapping)
    
    # Видаляємо Київ та Севастополь
    df = df[df['Area_ID'] != 0]
    
    # Переставляємо колонку Area_ID на перше місце для краси
    cols = ['Area_ID', 'Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']
    df = df[cols]
    
    print("Готово!")
    return df

# Викликаємо функцію і записуємо результат у змінну df
df = create_dataframe_and_clean()

# Виводимо перші 5 рядків таблиці
display(df.head())

Починаємо збирання та очищення даних...
Готово!


,Area_ID,Year,Week,SMN,SMT,VCI,TCI,VHI
0,21,1982,1,0.059,258.24,51.11,48.78,49.95
1,21,1982,2,0.063,261.53,55.89,38.20,47.04
2,21,1982,3,0.063,263.45,57.30,32.69,44.99
3,21,1982,4,0.061,265.10,53.96,28.62,41.29
4,21,1982,5,0.058,266.42,46.87,28.57,37.72


Ряд VHI для області за вказаний рік

In [3]:
def get_vhi_series(df, province_id, year):
    # Фільтруємо таблицю, шукаємо збіг по області та по року
    result = df[(df['Area_ID'] == province_id) & (df['Year'] == year)][['Week', 'VHI']]
    return result

print("Ряд VHI для області 1 (Вінницька) за 2020 рік:")
# .head(10) виведе лише перші 10 тижнів
display(get_vhi_series(df, province_id=1, year=2020).head(10))

Ряд VHI для області 1 (Вінницька) за 2020 рік:


,Week,VHI
35516,1,40.92
35517,2,43.19
35518,3,44.74
35519,4,45.29
35520,5,44.80
35521,6,43.92
35522,7,43.10
35523,8,42.88
35524,9,43.71
35525,10,45.61


Ряд VHI за вказаний діапазон років для вказаних областей

In [5]:
def get_vhi_range(df, provinces, year_start, year_end):
    # Фільтруємо за списком областей та діапазоном років
    result = df[(df['Area_ID'].isin(provinces)) & (df['Year'] >= year_start) & (df['Year'] <= year_end)]
    return result[['Year', 'Week', 'Area_ID', 'VHI']]

print("VHI для областей 1 (Вінницька) та 2 (Волинська) за 2018-2020 роки:")
display(get_vhi_range(df, provinces=[1, 2], year_start=2018, year_end=2020).head(10))

VHI для областей 1 (Вінницька) та 2 (Волинська) за 2018-2020 роки:


,Year,Week,Area_ID,VHI
35412,2018,1,1,46.61
35413,2018,2,1,49.50
35414,2018,3,1,52.71
35415,2018,4,1,53.30
35416,2018,5,1,51.76
35417,2018,6,1,49.91
35418,2018,7,1,47.45
35419,2018,8,1,46.65
35420,2018,9,1,48.45
35421,2018,10,1,49.56


Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани

In [6]:
def get_vhi_stats(df, provinces, years):
    # Фільтруємо дані
    filtered_df = df[(df['Area_ID'].isin(provinces)) & (df['Year'].isin(years))]

    # Рахуємо статистику
    stats = {
        'Min VHI': filtered_df['VHI'].min(),
        'Max VHI': filtered_df['VHI'].max(),
        'Mean VHI': filtered_df['VHI'].mean(),
        'Median VHI': filtered_df['VHI'].median()
    }
    return pd.DataFrame([stats])

print("Статистика VHI для областей 1 та 5 за 2020 та 2021 роки:")
display(get_vhi_stats(df, provinces=[1, 5], years=[2020, 2021]))

Статистика VHI для областей 1 та 5 за 2020 та 2021 роки:


,Min VHI,Max VHI,Mean VHI,Median VHI
0,33.55,71.59,48.625529,46.895
